# 🏨 🏬 Modélisation de l'algorithme de prédiction de la consommation d'énergie des batâments non résidentiels de la ville de Seattle

### 📃 Context du document

La ville de seattle a un objectif de neutralité carbone d'ici 2050. 
En 2016, elle a effectué un relevé de tous les batâments de ville afin de connaitre leur consommation ainsi que leur émission de CO2. 

La ville souhaiterait actualiser ces données, mais ce genre de relevé est très couteux. 
C'est pourquoi nous souhaitons développer un modèle prédictif qui permettra de prédire la consommation d'énergie. 

Nous avons au préalable effectué une analyse exploratoire qui nous a permis de mieux appréhender les données. 
A présent, nous allons passer à la deuxième étape, qui consiste à préparer les données pour l'entrainement du modèle.

### 📒 Import des modules

In [12]:
# Manipulation
import pandas as pd
import numpy as np

#Selection
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV, 
    cross_validate,
)
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error 
from sklearn.inspection import permutation_importance

#Preprocess
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, OrdinalEncoder

#Modèles
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor

In [13]:
pd.options.display.float_format = '{:.2f}'.format

### 🔍 Lecture et filtrage du dataset

In [14]:
df = pd.read_csv("2016_Building_Energy_Benchmarking.csv")

residential_building = ['Mixed Use Property',
                        'Residence Hall',
                        'High-Rise Multifamily',
                        'Low-Rise Multifamily',
                        'Other',
                        'Office',
                        'Mid-Rise Multifamily']

df = df[~df["PrimaryPropertyType"].isin(residential_building)]

### ⚙️ Features Engineering

In [21]:
col_to_keep = ['PrimaryPropertyType','YearBuilt', 'NumberofBuildings',
               'NumberofFloors', 'PropertyGFAParking','PropertyGFATotal', 
               'SiteEnergyUseWN(kBtu)']

df = df.loc[:,col_to_keep]

In [27]:
df.isna().sum()

PrimaryPropertyType      0
YearBuilt                0
NumberofBuildings        1
NumberofFloors           0
PropertyGFAParking       0
PropertyGFATotal         0
SiteEnergyUseWN(kBtu)    0
dtype: int64

In [34]:
df.describe()

,YearBuilt,NumberofBuildings,NumberofFloors,PropertyGFAParking,PropertyGFATotal,SiteEnergyUseWN(kBtu)
count,1305.00,1305.00,1305.00,1305.00,1305.00,1305.00
mean,1962.58,1.19,5.49,12902.58,124861.22,7555148.10
std,31.89,3.21,7.17,41971.25,322798.20,18647220.04
min,1900.00,1.00,1.00,0.00,16300.00,0.00
25%,1937.00,1.00,2.00,0.00,31357.00,1266706.88
50%,1966.00,1.00,4.00,0.00,52110.00,2626698.25
75%,1988.00,1.00,5.00,0.00,112896.00,7007726.00
max,2015.00,111.00,100.00,512608.00,9320156.00,296671744.00


In [33]:
# Remplacement des valeurs nulles de la variable énergie (médiane de la catégorie du bâtiment)
df['SiteEnergyUseWN(kBtu)'] = df['SiteEnergyUseWN(kBtu)'].fillna(
    df.groupby('PrimaryPropertyType')['SiteEnergyUseWN(kBtu)'].transform('median')
)

# Remplacement de la valeur nulle de la variable nombre de batiment (par défaut Nb bat = 1)
df['NumberofBuildings'] = df['NumberofBuildings'].fillna(1)

# Retraitement des NB bat = 0 par Nb bat = 1
df.loc[df['NumberofBuildings'] == 0, 'NumberofBuildings'] = 1

# Retraitement des Nb floor, on considère que le RDC est le 1er étage
df['NumberofFloors'] += 1

In [16]:
# Année du bâtiment
df['Building_Age'] = 2016 - df['YearBuilt']